# 01 Baseline Evaluation and Dataset Preparation

> Nemotron SFT-RL 课程大作业 · 第 1 个 Notebook
>
> 本 Notebook 是为 `datawhalechina/llm-algo-leetcode` 后训练优化路线准备的参考实现。它先固定真实数据身份、分类、划分和泄漏审计，再提供一个默认关闭的 Qwen3/vLLM Baseline 入口，以及可选的数据合成与筛查 Pilot。

**当前证据状态**：数据读取、分类、清洗、IID 划分、审计和合成 smoke test 可在无模型下载条件下验证；Qwen3 模型权重和 vLLM GPU 推理只有在目标 RTX 5090 与官方 vLLM Profile 准备好后才运行。Notebook 不在执行过程中安装依赖。


## 学习目标

完成本节后，你应当能够：

1. 从官方 `train.csv` 的实际样本归纳可审计的题型分类，而不是依赖旧统计。
2. 将 Equation Numeric 与 Equation Symbolic 分开，并保留所有被排除样本的审计记录。
3. 在固定随机种子下生成 Train / Validation / Holdout Test，并检查精确重复、规范化重复和跨集合污染。
4. 理解原始模型 Baseline 的边界：原始 Prompt、官方 Chat Template、统一解码和确定性答案验证。
5. 在已确认的先验范围内构造小型合成样本；当规则定义缺失时，记录阻断原因而不是猜测比赛规则。


## 1. 实验约束与运行配置

首版只研究三个真实题型：Gravity、Equation Numeric、Bit Manipulation。Gravity 是能力保持控制组，Equation Numeric 和 Bit Manipulation 是后续 RL 的主要任务。模型固定为 `Qwen/Qwen3-4B-Thinking-2507`，训练和 vLLM Rollout 使用 Datawhale 官方隔离 Profile。

`RUN_BASELINE=False` 是安全默认值：它不会下载模型或占用 GPU。准备好官方 vLLM Profile 后，将其改为 `True`，并先运行 `smoke`。


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_CANDIDATES = [
    PROJECT_ROOT / "train.csv",
    PROJECT_ROOT.parent / "train.csv",
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(f"未找到 train.csv，检查路径：{RAW_CANDIDATES}")

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = OUTPUT_DIR / "splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
RUN_MODE = "smoke"                 # smoke / full
MAX_SAMPLES_PER_TASK = 20
SEED = 42
RUN_BASELINE = False               # 默认不下载模型
RUN_SYNTHETIC = False              # True 时才生成每类 100 条 Pilot
SYNTHETIC_SAMPLES_PER_TASK = 100
SOURCE_DATASET = "NVIDIA Nemotron Model Reasoning Challenge / train.csv"

TARGET_TASKS = {"gravity", "equation_numeric", "bit_manipulation"}
print({"raw_path": str(RAW_PATH), "output_dir": str(OUTPUT_DIR), "model_id": MODEL_ID, "run_baseline": RUN_BASELINE})


## 2. 读取并核验官方数据

这里使用字符串读取，避免 ID、二进制答案或带前导零的数值被自动转换。`source_row` 是原始 CSV 中的行号（表头为第 1 行），后续所有产物都保留它和原始 `id`。


In [ ]:
raw_df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
required_columns = ["id", "prompt", "answer"]
missing_columns = [c for c in required_columns if c not in raw_df.columns]
if missing_columns:
    raise ValueError(f"缺少字段：{missing_columns}")
raw_df = raw_df[required_columns].copy()
raw_df.insert(0, "source_row", np.arange(2, len(raw_df) + 2, dtype=int))
raw_df["source_dataset"] = SOURCE_DATASET

empty_counts = {c: int((raw_df[c].str.len() == 0).sum()) for c in required_columns}
raw_profile = {
    "sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "rows": int(len(raw_df)),
    "columns": list(raw_df.columns),
    "dtypes": {c: str(raw_df[c].dtype) for c in raw_df.columns},
    "empty_counts": empty_counts,
    "null_counts": {c: int(raw_df[c].isna().sum()) for c in raw_df.columns},
    "duplicate_id_rows": int(raw_df["id"].duplicated(keep=False).sum()),
    "duplicate_id_repeats": int(raw_df["id"].duplicated().sum()),
    "duplicate_prompt_rows": int(raw_df["prompt"].duplicated(keep=False).sum()),
    "duplicate_prompt_repeats": int(raw_df["prompt"].duplicated().sum()),
}
print(json.dumps(raw_profile, ensure_ascii=False, indent=2))
print("\n首行字段：", raw_df.iloc[0][required_columns].to_dict())


## 3. 题型分类：使用精确模板，再细分 Equation

官方数据的六个首句模板可以被精确识别：Gravity、Bit Manipulation、Equation、Unit Conversion、Encryption 和 Numeral System。Equation 模板内部必须再分：含数字的样本是 Equation Numeric，完全由符号构成的样本是 Equation Symbolic。

这里不使用宽泛的 `"equation" in prompt` 规则。遇到没有匹配模板的样本会进入 `unknown`，不会静默丢弃。Numeric 样本还会根据 Query 运算符是否在示例中出现，标记 `deduce` 或 `guess`。


In [ ]:
TEMPLATE_PREFIXES = [
    ("gravity", "In Alice's Wonderland, the gravitational constant has been secretly changed.", "exact_gravity_prefix"),
    ("bit_manipulation", "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.", "exact_bit_prefix"),
    ("equation", "In Alice's Wonderland, a secret set of transformation rules is applied to equations.", "exact_equation_prefix"),
    ("unit_conversion", "In Alice's Wonderland, a secret unit conversion is applied to measurements.", "exact_unit_prefix"),
    ("encryption", "In Alice's Wonderland, secret encryption rules are used on text.", "exact_encryption_prefix"),
    ("numeral_system", "In Alice's Wonderland, numbers are secretly converted into a different numeral system.", "exact_numeral_prefix"),
]


def normalize_prompt(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"0b[01]+", "<bin>", text)
    text = re.sub(r"[-+]?\d+(?:\.\d+)?", "<num>", text)
    text = re.sub(r"\s+", " ", text)
    return text


def prompt_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def query_operator_and_example_operators(prompt: str) -> tuple[str | None, set[str]]:
    example_ops = set(re.findall(r"(?m)^\s*\d+\s*(.)\s*\d+\s*=", prompt))
    query = re.search(r"Now, determine the result for:\s*([0-9]+)\s*(.)\s*([0-9]+)", prompt)
    if query is None:
        return None, example_ops
    return query.group(2), example_ops


def classify_prompt(prompt: str) -> dict[str, object]:
    prompt = str(prompt)
    for base_label, prefix, rule in TEMPLATE_PREFIXES:
        if prompt.startswith(prefix):
            if base_label == "equation":
                if re.search(r"\d", prompt):
                    query_op, example_ops = query_operator_and_example_operators(prompt)
                    subtype = "deduce" if query_op in example_ops else "guess"
                    if query_op is None:
                        subtype = "undetermined"
                    return {
                        "task_type": "equation_numeric",
                        "equation_subtype": subtype,
                        "classification_confidence": "high",
                        "classification_rule": "exact_equation_prefix_plus_digit_presence",
                        "query_operator": query_op or "",
                        "example_operators": "".join(sorted(example_ops)),
                    }
                return {
                    "task_type": "equation_symbolic",
                    "equation_subtype": "symbolic",
                    "classification_confidence": "high",
                    "classification_rule": "exact_equation_prefix_plus_no_digits",
                    "query_operator": "",
                    "example_operators": "",
                }
            return {
                "task_type": base_label,
                "equation_subtype": "",
                "classification_confidence": "high",
                "classification_rule": rule,
                "query_operator": "",
                "example_operators": "",
            }
    return {
        "task_type": "unknown",
        "equation_subtype": "",
        "classification_confidence": "low",
        "classification_rule": "no_registered_template_prefix",
        "query_operator": "",
        "example_operators": "",
    }


classification_records = []
for row in raw_df.itertuples(index=False):
    result = classify_prompt(row.prompt)
    normalized = normalize_prompt(row.prompt)
    classification_records.append({
        "source_row": int(row.source_row),
        "source_dataset": row.source_dataset,
        "id": row.id,
        "original_prompt": row.prompt,
        "original_answer": row.answer,
        "prompt_hash": prompt_hash(row.prompt),
        "normalized_prompt_hash": prompt_hash(normalized),
        "has_digit": bool(re.search(r"\d", row.prompt)),
        "prompt_characters": int(len(row.prompt)),
        "answer_characters": int(len(row.answer)),
        **result,
        "kept_for_course": result["task_type"] in TARGET_TASKS,
        "exclusion_reason": "" if result["task_type"] in TARGET_TASKS else f"excluded_{result['task_type']}",
    })

audit_df = pd.DataFrame(classification_records)
print(audit_df["task_type"].value_counts(dropna=False).to_string())
print("\nEquation Numeric subtype:")
print(audit_df.loc[audit_df.task_type == "equation_numeric", "equation_subtype"].value_counts(dropna=False).to_string())


In [ ]:
print("各分类抽样（只显示 Prompt 首句和答案，不修改原始数据）：")
for task_type, group in raw_df.assign(task_type=audit_df["task_type"]).groupby("task_type", sort=True):
    print(f"\n### {task_type}: n={len(group)}")
    sample = group.sample(n=min(2, len(group)), random_state=SEED)
    for row in sample.itertuples(index=False):
        print({"id": row.id, "prompt_head": row.prompt.splitlines()[0], "answer": row.answer})

unknown_rows = audit_df[audit_df.task_type == "unknown"]
if len(unknown_rows):
    print("\nUNKNOWN 样本必须人工复核：")
    print(unknown_rows[["source_row", "id", "classification_rule"]].head(20).to_string(index=False))
else:
    print("\n没有未知模板；仍保留 unknown 分类路径以防数据版本改变。")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = audit_df["task_type"].value_counts().sort_values(ascending=True)
counts.plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Raw task distribution (exact template classification)")
ax.set_xlabel("Number of samples")
ax.set_ylabel("Task type")
for i, value in enumerate(counts):
    ax.text(value + max(counts) * 0.01, i, str(value), va="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "raw_task_distribution.png", dpi=150)
plt.show()


## 4. 清洗主线：只保留三个正式题型

清洗只保留 `gravity`、`equation_numeric` 和 `bit_manipulation`。`equation_symbolic`、Unit Conversion、Encryption、Numeral System 和 Unknown 都保留在 `classification_audit.csv` 中，并带有排除原因。

`real_cleaned.csv` 仍保留原始 ID、原始 Prompt、原始 Answer 和来源行号，后续训练数据可以从它追溯回官方文件。


In [ ]:
raw_with_classification = raw_df.join(audit_df.set_index("source_row"), on="source_row", rsuffix="_audit")
cleaned_df = raw_with_classification[raw_with_classification["task_type"].isin(sorted(TARGET_TASKS))].copy()
cleaned_columns = [
    "source_row", "source_dataset", "id", "prompt", "answer", "task_type", "equation_subtype",
    "query_operator", "example_operators", "prompt_hash", "normalized_prompt_hash",
    "classification_confidence", "classification_rule",
]
audit_path = OUTPUT_DIR / "classification_audit.csv"
cleaned_path = OUTPUT_DIR / "real_cleaned.csv"
audit_df.to_csv(audit_path, index=False)
cleaned_df[cleaned_columns].to_csv(cleaned_path, index=False)

cleaning_report = {
    "raw_rows": int(len(raw_df)),
    "kept_rows": int(len(cleaned_df)),
    "removed_rows": int(len(raw_df) - len(cleaned_df)),
    "kept_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "removed_counts": {k: int(v) for k, v in audit_df.loc[~audit_df.kept_for_course, "task_type"].value_counts().to_dict().items()},
    "audit_path": str(audit_path),
    "cleaned_path": str(cleaned_path),
}
print(json.dumps(cleaning_report, ensure_ascii=False, indent=2))


## 5. 划分与 Leakage Audit

默认划分是固定种子的**分层 IID Holdout**：每个正式题型分别按 80% / 10% / 10% 划分。它能提供同分布泛化观察，但不能自动证明不存在同源题目泄漏。

本数据没有公开的规则实例 ID、模板组 ID 或生成种子。我们会审计完全重复 Prompt 和把数字/二进制字面量替换后的规范化重复；如果规范化重复跨集合，只报告风险，不把它伪装成 Structural OOD 证据。Structural OOD 需要后续补充可靠的规则组合或结构家族字段后再做。


In [ ]:
def make_stratified_split(frame: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    assignments = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        indices = group.index.to_numpy(copy=True)
        rng.shuffle(indices)
        n = len(indices)
        n_validation = max(1, int(round(n * 0.10)))
        n_test = max(1, int(round(n * 0.10)))
        for index in indices[: n - n_validation - n_test]:
            assignments[index] = "train"
        for index in indices[n - n_validation - n_test : n - n_test]:
            assignments[index] = "validation"
        for index in indices[n - n_test :]:
            assignments[index] = "holdout_test"
    result = frame.copy()
    result["split"] = pd.Series(assignments)
    if result["split"].isna().any():
        raise AssertionError("存在未分配 split 的样本")
    return result

split_df = make_stratified_split(cleaned_df, seed=SEED)
for split_name, split_group in split_df.groupby("split", sort=True):
    split_group[cleaned_columns + ["split"]].to_csv(SPLIT_DIR / f"{split_name}.csv", index=False)


def cross_split_duplicate_counts(frame: pd.DataFrame, key: str) -> dict[str, object]:
    owners = frame.groupby(key)["split"].agg(lambda values: sorted(set(values)))
    leaked = owners[owners.map(len) > 1]
    return {
        "key": key,
        "duplicate_keys": int((owners.map(len) > 1).sum()),
        "rows_in_cross_split_keys": int(frame[frame[key].isin(leaked.index)].shape[0]),
        "examples": [{"key": str(k), "splits": v} for k, v in leaked.head(5).items()],
    }

split_counts = (
    split_df.groupby(["split", "task_type"], sort=True).size().unstack(fill_value=0).astype(int)
)
leakage_audit = {
    "exact_id": cross_split_duplicate_counts(split_df, "id"),
    "exact_prompt": cross_split_duplicate_counts(split_df, "prompt_hash"),
    "normalized_prompt": cross_split_duplicate_counts(split_df, "normalized_prompt_hash"),
    "group_aware_split": {
        "status": "not_available",
        "reason": "原始 CSV 未提供可靠的规则实例、生成种子或结构家族字段；规范化 Prompt 只能用于审计，不能作为同源组的充分证明。",
    },
}
print(split_counts.to_string())
print(json.dumps(leakage_audit, ensure_ascii=False, indent=2))


In [ ]:
def prompt_token_proxy(text: str) -> int:
    # 这是不依赖模型 Tokenizer 的审计量，不应冒充 Qwen3 token 数。
    return len(re.findall(r"\w+|[^\w\s]", str(text), flags=re.UNICODE))

split_df["prompt_token_proxy"] = split_df["prompt"].map(prompt_token_proxy)
rule_distribution = {
    "task_type": {k: int(v) for k, v in split_df.task_type.value_counts().to_dict().items()},
    "equation_subtype": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "equation_subtype"].value_counts().to_dict().items()},
    "equation_query_operator": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "query_operator"].value_counts().to_dict().items()},
}

dataset_statistics = {
    "schema_version": "nemotron-course-dataset/v1",
    "source": {"path": str(RAW_PATH.relative_to(PROJECT_ROOT)), "dataset": SOURCE_DATASET, "sha256": raw_profile["sha256"]},
    "seed": SEED,
    "raw_profile": raw_profile,
    "classification_counts": {k: int(v) for k, v in audit_df.task_type.value_counts().to_dict().items()},
    "cleaned_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "split_counts": {
        split: {task: int(value) for task, value in row.items()}
        for split, row in split_counts.iterrows()
    },
    "rule_distribution": rule_distribution,
    "length_summary": {
        split: {
            "rows": int(len(group)),
            "prompt_char_mean": float(group.prompt.str.len().mean()),
            "prompt_token_proxy_mean": float(group.prompt_token_proxy.mean()),
            "answer_char_mean": float(group.answer.str.len().mean()),
        }
        for split, group in split_df.groupby("split", sort=True)
    },
    "leakage_audit": leakage_audit,
    "limitations": [
        "prompt_token_proxy 不是 Qwen3 Tokenizer 的真实 token 数；真实 token 统计需在官方训练环境加载 tokenizer 后补充。",
        "没有可靠的规则实例/结构家族键，因此当前只提供 IID Holdout，不宣称 Structural OOD 已完成。",
        "规范化 Prompt 重复是风险信号，不等同于同源题目证明。",
    ],
}
(OUTPUT_DIR / "dataset_statistics.json").write_text(json.dumps(dataset_statistics, ensure_ascii=False, indent=2), encoding="utf-8")
split_manifest = {
    "schema_version": "nemotron-course-split/v1",
    "seed": SEED,
    "ratios_requested": {"train": 0.8, "validation": 0.1, "holdout_test": 0.1},
    "method": "per-task stratified IID split",
    "structural_ood": "not_available_without_reliable_group_key",
    "files": {name: str((SPLIT_DIR / f"{name}.csv").relative_to(PROJECT_ROOT)) for name in ["train", "validation", "holdout_test"]},
    "leakage_audit": leakage_audit,
}
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({"statistics": str(OUTPUT_DIR / 'dataset_statistics.json'), "split_manifest": str(OUTPUT_DIR / 'split_manifest.json')}, ensure_ascii=False, indent=2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_counts = split_df.groupby(["split", "task_type"]).size().unstack(fill_value=0)
plot_counts.plot.bar(ax=ax, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Stratified IID split by task type")
ax.set_xlabel("Split")
ax.set_ylabel("Number of samples")
ax.legend(title="Task type")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "split_task_distribution.png", dpi=150)
plt.show()


## 6. Baseline：原始 Prompt、统一生成和确定性验证

Baseline 必须回答“未经 Prompt Engineering、SFT 或 RL 的 Qwen3 基座模型表现如何”。因此：

- 输入只使用原始比赛 Prompt；不注入课程先验规则库。
- 生成使用 Qwen3 官方 Chat Template，并在模型支持时显式开启 Thinking。
- 评测输出保留原文、提取答案、提取状态、生成 token 数和耗时。
- 不把没有运行的分数写成结果。

答案提取器只负责把模型回答映射到候选答案；Verifier 再按题型比较。对无法提取或格式不合法的输出，计入 Invalid Output，而不是当作错误答案静默吞掉。


In [ ]:
def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I)
    return text.strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?:final|answer)>\s*(.*?)\s*</(?:final|answer)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1]), "tagged"
    if task_type == "bit_manipulation":
        matches = re.findall(r"(?<![01])[01]{8}(?![01])", text)
        if matches:
            return matches[-1], "binary_fallback"
    if task_type in {"gravity", "equation_numeric"}:
        matches = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", text)
        if matches:
            return matches[-1], "numeric_fallback"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        return normalize_answer_text(lines[-1]), "last_nonempty_line"
    return None, "empty"


def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text = normalize_answer_text(predicted)
    expected_text = normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p = predicted_text.removeprefix("0b")
        e = expected_text.removeprefix("0b")
        if not re.fullmatch(r"[01]{8}", p):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        try:
            return Decimal(predicted_text) == Decimal(expected_text), "decimal_equivalence"
        except InvalidOperation:
            return False, "invalid_numeric_format"
    return predicted_text == expected_text, "exact_text"


def summarize_baseline_records(records: list[dict[str, object]]) -> dict[str, object]:
    frame = pd.DataFrame(records)
    if frame.empty:
        return {"records": 0, "status": "no_records"}
    summaries = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        summaries[task_type] = {
            "records": int(len(group)),
            "accuracy": float(group["correct"].mean()),
            "answer_extraction_success_rate": float(group["extraction_success"].mean()),
            "invalid_output_rate": float((~group["extraction_success"]).mean()),
            "average_generation_tokens": float(group["generation_tokens"].mean()),
            "average_inference_time_seconds": float(group["inference_time_seconds"].mean()),
            "total_inference_time_seconds": float(group["inference_time_seconds"].sum()),
        }
    task_accuracy = frame.groupby("task_type")["correct"].mean()
    return {
        "records": int(len(frame)),
        "overall_accuracy_sample_weighted": float(frame["correct"].mean()),
        "macro_accuracy": float(task_accuracy.mean()),
        "by_task": summaries,
    }


In [ ]:
def render_qwen3_prompt(tokenizer, original_prompt: str) -> str:
    messages = [{"role": "user", "content": original_prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        # 兼容旧版 Transformers：仍使用官方模板，但不传新参数。
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def run_vllm_baseline(
    evaluation_frame: pd.DataFrame,
    model_id: str = MODEL_ID,
    max_new_tokens: int = 512,
    output_path: Path = OUTPUT_DIR / "baseline_results.jsonl",
) -> dict[str, object]:
    """在官方独立 vLLM Profile 中运行 Baseline；函数本身不安装依赖。"""
    try:
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams
    except ImportError as exc:
        return {"status": "blocked_missing_vllm_profile", "reason": repr(exc)}

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    prompts = [render_qwen3_prompt(tokenizer, p) for p in evaluation_frame["prompt"]]
    llm = LLM(model=model_id, tensor_parallel_size=1, gpu_memory_utilization=0.85)
    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_new_tokens)
    started = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    elapsed_total = time.perf_counter() - started

    records = []
    for row, request_output in zip(evaluation_frame.itertuples(index=False), outputs):
        generated = request_output.outputs[0]
        text = generated.text
        extracted, extraction_method = extract_final_answer(text, row.task_type)
        correct, verification_method = verify_answer(row.task_type, extracted, row.answer)
        token_ids = getattr(generated, "token_ids", ()) or ()
        records.append({
            "id": row.id,
            "source_row": int(row.source_row),
            "task_type": row.task_type,
            "prompt": row.prompt,
            "expected_answer": row.answer,
            "raw_output": text,
            "extracted_answer": extracted,
            "extraction_method": extraction_method,
            "verification_method": verification_method,
            "extraction_success": extracted is not None,
            "correct": bool(correct),
            "generation_tokens": int(len(token_ids)),
            "inference_time_seconds": float(elapsed_total / max(len(outputs), 1)),
        })
    output_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n", encoding="utf-8")
    summary = summarize_baseline_records(records)
    (OUTPUT_DIR / "baseline_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": "completed", "summary": summary, "results_path": str(output_path)}


if RUN_BASELINE:
    baseline_frame = split_df[split_df["split"] == "holdout_test"].copy()
    if RUN_MODE == "smoke":
        baseline_frame = pd.concat([
            group.head(MAX_SAMPLES_PER_TASK) for _, group in baseline_frame.groupby("task_type", sort=True)
        ], ignore_index=True)
    baseline_status = run_vllm_baseline(baseline_frame)
else:
    baseline_status = {
        "status": "skipped_by_default",
        "reason": "RUN_BASELINE=False；未下载模型、未占用 GPU、未生成虚构分数。",
        "next_action": "在官方 INF-VLLM Profile 和 RTX 5090 可用时改为 True，并先运行 smoke。",
    }
print(json.dumps(baseline_status, ensure_ascii=False, indent=2))


## 7. 可选数据工程：先验边界和合成 Pilot

合成数据不是主线前置。所有真实数据先完成划分，验证集和 Holdout Test 不进入合成过程。

综述中明确的先验包括：

- Gravity：`d = 0.5 * g * t^2`、单位和精度约定。
- Bit Manipulation：8-bit 的 SHL、SHR、ROT、AND、OR、XOR、NOT，以及有限 Boolean DSL。
- Equation Numeric：只确认“有候选运算规则库”这一教学资产，但当前综述没有列出候选运算符的完整数学定义、参数域和可识别性约束。

因此本 Notebook 只实现前两类的教学 Pilot；Equation Numeric 生成器会明确阻断并列出缺失定义。Pilot 使用独立的生成表示和验证表示，保留拒绝样本的原因。


In [ ]:
GRAVITY_PRIOR = {
    "formula": "distance = 0.5 * g * t^2",
    "units": {"t": "seconds", "distance": "meters"},
    "answer_precision": "two_decimal_places",
}
BIT_PRIOR = {
    "width": 8,
    "operations": ["SHL", "SHR", "ROTL", "ROTR", "AND", "OR", "XOR", "NOT"],
    "mask": 255,
}
MISSING_EQUATION_NUMERIC_DEFINITIONS = [
    "候选运算符的完整集合及每个运算符的数学定义",
    "操作数范围、输出编码和异常/边界规则",
    "Deduce/Guess 的可识别性判定算法和全局约束",
]
print(json.dumps({"gravity_prior": GRAVITY_PRIOR, "bit_prior": BIT_PRIOR, "equation_numeric_generator": {"status": "blocked", "missing": MISSING_EQUATION_NUMERIC_DEFINITIONS}}, ensure_ascii=False, indent=2))


In [ ]:
def gravity_answer(t: float, g: float) -> str:
    return f"{0.5 * g * t * t:.2f}"


def generate_gravity_sample(rng: np.random.Generator, examples: int = 3) -> dict[str, object]:
    g = float(rng.uniform(9.5, 19.5))
    times = [round(float(rng.uniform(0.6, 5.0)), 2) for _ in range(examples + 1)]
    lines = [
        f"For t = {t:.2f}s, distance = {gravity_answer(t, g)} m"
        for t in times[:examples]
    ]
    query_t = times[-1]
    prompt = (
        "In a course-provided gravity prior, use d = 0.5*g*t^2.\n"
        + "\n".join(lines)
        + f"\nNow, calculate the falling distance for t = {query_t:.2f}s."
    )
    return {
        "task_type": "gravity",
        "prompt": prompt,
        "answer": gravity_answer(query_t, g),
        "trace": {"formula": GRAVITY_PRIOR["formula"], "hidden_g": g, "query_t": query_t},
        "difficulty": "easy_control",
        "generator": "gravity_prior_v1",
    }


def verify_gravity_sample(sample: dict[str, object]) -> tuple[bool, str]:
    pairs = re.findall(r"For t = ([0-9.]+)s, distance = ([0-9.]+) m", sample["prompt"])
    if len(pairs) < 2:
        return False, "insufficient_examples"
    estimates = [Decimal(d) / (Decimal("0.5") * Decimal(t) * Decimal(t)) for t, d in pairs]
    if max(estimates) - min(estimates) > Decimal("0.10"):
        return False, "inconsistent_hidden_g"
    query = re.search(r"t = ([0-9.]+)s\.", sample["prompt"].split("Now,")[-1])
    if query is None:
        return False, "missing_query"
    expected = gravity_answer(float(query.group(1)), float(sum(estimates) / len(estimates)))
    return expected == str(sample["answer"]), "independent_formula_check"


In [ ]:
MASK8 = 0xFF


def compute_bit_tree(node: tuple) -> int:
    kind = node[0]
    if kind == "lit":
        return int(node[1]) & MASK8
    if kind in {"SHL", "SHR", "ROTL", "ROTR"}:
        value, shift = compute_bit_tree(node[1]), int(node[2]) % 8
        if kind == "SHL":
            return (value << shift) & MASK8
        if kind == "SHR":
            return (value >> shift) & MASK8
        if kind == "ROTL":
            return ((value << shift) | (value >> (8 - shift))) & MASK8 if shift else value
        return ((value >> shift) | (value << (8 - shift))) & MASK8 if shift else value
    if kind == "NOT":
        return (~compute_bit_tree(node[1])) & MASK8
    left, right = compute_bit_tree(node[1]), compute_bit_tree(node[2])
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def render_bit_tree(node: tuple) -> str:
    kind = node[0]
    if kind == "lit":
        return f"0b{int(node[1]) & MASK8:08b}"
    if kind in {"SHL", "SHR", "ROTL", "ROTR"}:
        return f"{kind}({render_bit_tree(node[1])}, {int(node[2])})"
    if kind == "NOT":
        return f"NOT({render_bit_tree(node[1])})"
    return f"{kind}({render_bit_tree(node[1])}, {render_bit_tree(node[2])})"


def random_bit_tree(rng: np.random.Generator, depth: int = 2) -> tuple:
    if depth <= 0:
        return ("lit", int(rng.integers(0, 256)))
    kind = str(rng.choice(["SHL", "SHR", "ROTL", "ROTR", "AND", "OR", "XOR", "NOT"]))
    left = random_bit_tree(rng, depth - 1)
    if kind == "NOT":
        return (kind, left)
    if kind in {"SHL", "SHR", "ROTL", "ROTR"}:
        return (kind, left, int(rng.integers(0, 8)))
    return (kind, left, random_bit_tree(rng, depth - 1))


def instantiate_bit_tree(tree: tuple, value: int) -> tuple:
    if tree[0] == "lit":
        return ("lit", int(value))
    if tree[0] in {"NOT", "SHL", "SHR", "ROTL", "ROTR"}:
        return (tree[0], instantiate_bit_tree(tree[1], value), *tree[2:])
    return (tree[0], instantiate_bit_tree(tree[1], value), tree[2])


def generate_bit_sample(rng: np.random.Generator, examples: int = 3) -> dict[str, object]:
    tree = random_bit_tree(rng, depth=2)
    example_values = [int(rng.integers(0, 256)) for _ in range(examples)]
    query_value = int(rng.integers(0, 256))
    expression_lines = []
    for value in example_values:
        example_tree = instantiate_bit_tree(tree, value)
        expression_lines.append(
            f"{render_bit_tree(example_tree)} = 0b{compute_bit_tree(example_tree):08b}"
        )
    query_tree = instantiate_bit_tree(tree, query_value)
    prompt = (
        "Use the course-provided 8-bit Boolean DSL. Operators are SHL, SHR, ROTL, ROTR, AND, OR, XOR and NOT; every result is masked to 8 bits.\n"
        + "\n".join(expression_lines)
        + f"\nNow, determine the result for: {render_bit_tree(query_tree)}"
    )
    return {
        "task_type": "bit_manipulation",
        "prompt": prompt,
        "answer": f"{compute_bit_tree(query_tree):08b}",
        "trace": {"expression": render_bit_tree(query_tree), "width": 8},
        "difficulty": "depth_2",
        "generator": "bit_dsl_prior_v1",
    }


def parse_bit_expression(expr: str, position: int = 0) -> tuple[int, int]:
    expr = expr.strip()
    position += len(expr[position:]) - len(expr[position:].lstrip())
    if expr.startswith("0b", position):
        match = re.match(r"0b([01]{8})", expr[position:])
        if not match:
            raise ValueError("invalid_literal")
        return int(match.group(1), 2), position + match.end()
    call = re.match(r"(SHL|SHR|ROTL|ROTR|AND|OR|XOR|NOT)\(", expr[position:])
    if not call:
        raise ValueError("invalid_operation")
    op = call.group(1)
    pos = position + call.end()
    left, pos = parse_bit_expression(expr, pos)
    if op == "NOT":
        if expr[pos:].lstrip().startswith(")"):
            pos = pos + len(expr[pos:]) - len(expr[pos:].lstrip()) + 1
            return (~left) & MASK8, pos
        raise ValueError("invalid_not")
    if op in {"SHL", "SHR", "ROTL", "ROTR"}:
        comma = expr.find(",", pos)
        close = expr.find(")", comma + 1)
        if comma < 0 or close < 0:
            raise ValueError("invalid_shift")
        shift = int(expr[comma + 1 : close].strip()) % 8
        if op == "SHL": value = (left << shift) & MASK8
        elif op == "SHR": value = (left >> shift) & MASK8
        elif op == "ROTL": value = ((left << shift) | (left >> (8 - shift))) & MASK8 if shift else left
        else: value = ((left >> shift) | (left << (8 - shift))) & MASK8 if shift else left
        return value, close + 1
    if expr[pos:].lstrip().startswith(","):
        pos = pos + len(expr[pos:]) - len(expr[pos:].lstrip()) + 1
    else:
        raise ValueError("missing_comma")
    right, pos = parse_bit_expression(expr, pos)
    if not expr[pos:].lstrip().startswith(")"):
        raise ValueError("missing_close")
    pos = pos + len(expr[pos:]) - len(expr[pos:].lstrip()) + 1
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[op] & MASK8, pos


def verify_bit_sample(sample: dict[str, object]) -> tuple[bool, str]:
    expression = sample["trace"]["expression"]
    try:
        value, position = parse_bit_expression(expression)
        if expression[position:].strip():
            return False, "trailing_expression"
        return f"{value:08b}" == str(sample["answer"]), "independent_dsl_parser_check"
    except (ValueError, TypeError, IndexError):
        return False, "invalid_dsl_expression"


In [ ]:
def generate_equation_numeric_sample(*args, **kwargs) -> dict[str, object]:
    raise NotImplementedError(
        "Equation Numeric 的候选运算符数学定义、参数域和可识别性约束尚未在综述中给出；"
        "请先补充课程 Prior Specification，再实现 Generator。"
    )


def screen_synthetic_sample(sample: dict[str, object], seen_hashes: set[str]) -> dict[str, object]:
    reasons = []
    schema_ok = all(sample.get(key) not in (None, "") for key in ["task_type", "prompt", "answer", "trace"])
    if not schema_ok:
        reasons.append("schema_invalid")
    exact_hash = prompt_hash(sample.get("prompt", ""))
    if exact_hash in seen_hashes:
        reasons.append("duplicate_prompt")
    if sample.get("task_type") == "gravity":
        answer_ok, answer_reason = verify_gravity_sample(sample)
        trace_ok = answer_ok
        identifiability = "pass" if answer_ok else "fail"
    elif sample.get("task_type") == "bit_manipulation":
        answer_ok, answer_reason = verify_bit_sample(sample)
        trace_ok = answer_ok
        # 当前没有完整的表达式搜索器，不能宣称全局唯一。
        identifiability = "not_evaluable_without_rule_search"
        reasons.append("identifiability_not_evaluable")
    else:
        answer_ok, answer_reason, trace_ok, identifiability = False, "unsupported_task", False, "blocked"
        reasons.append("unsupported_task")
    if not answer_ok:
        reasons.append("answer_verification_failed")
    if not trace_ok:
        reasons.append("trace_verification_failed")
    accepted = schema_ok and answer_ok and trace_ok and identifiability == "pass" and not any(r in reasons for r in ["duplicate_prompt"])
    return {
        "task_type": sample.get("task_type", ""),
        "prompt_hash": exact_hash,
        "schema_ok": schema_ok,
        "answer_ok": answer_ok,
        "answer_reason": answer_reason,
        "trace_ok": trace_ok,
        "identifiability": identifiability,
        "difficulty": sample.get("difficulty", ""),
        "accepted": accepted,
        "rejection_reasons": ";".join(dict.fromkeys(reasons)),
    }


def run_synthetic_pilot_smoke() -> dict[str, object]:
    rng = np.random.default_rng(SEED)
    samples = [generate_gravity_sample(rng, examples=3), generate_bit_sample(rng, examples=3)]
    blocked_equation = {
        "task_type": "equation_numeric",
        "prompt": "[blocked: Equation Numeric prior specification missing]",
        "answer": "",
        "trace": {},
        "difficulty": "blocked",
    }
    records = []
    seen = set()
    for sample in samples:
        record = screen_synthetic_sample(sample, seen)
        records.append(record)
        seen.add(record["prompt_hash"])
    records.append({
        "task_type": blocked_equation["task_type"],
        "prompt_hash": "",
        "schema_ok": False,
        "answer_ok": False,
        "answer_reason": "missing_rule_library",
        "trace_ok": False,
        "identifiability": "blocked",
        "difficulty": "blocked",
        "accepted": False,
        "rejection_reasons": "missing_equation_numeric_rule_library",
    })
    result_df = pd.DataFrame(records)
    result_df.to_csv(OUTPUT_DIR / "synthetic_pilot_screening.csv", index=False)
    summary = {
        "mode": "smoke",
        "generated": int(len(records)),
        "verified_answer_and_trace": int((result_df.answer_ok & result_df.trace_ok).sum()),
        "accepted": int(result_df.accepted.sum()),
        "rejected": int((~result_df.accepted).sum()),
        "by_task": {k: int(v) for k, v in result_df.task_type.value_counts().to_dict().items()},
        "rejection_reasons": {k: int(v) for k, v in Counter(";".join(result_df.rejection_reasons).split(";")).items() if k},
        "artifacts": {"screening": str((OUTPUT_DIR / "synthetic_pilot_screening.csv").relative_to(PROJECT_ROOT))},
    }
    (OUTPUT_DIR / "synthetic_statistics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary


synthetic_smoke_summary = run_synthetic_pilot_smoke()
print(json.dumps(synthetic_smoke_summary, ensure_ascii=False, indent=2))


## 8. 交付检查与未完成事项

本节把可复核的文件路径和当前证据状态集中展示。Baseline 的准确率只有在 `RUN_BASELINE=True` 且 vLLM 实际完成后才会出现；本次默认执行不会填入任何模型分数。


In [ ]:
expected_artifacts = [
    OUTPUT_DIR / "real_cleaned.csv",
    OUTPUT_DIR / "classification_audit.csv",
    OUTPUT_DIR / "dataset_statistics.json",
    OUTPUT_DIR / "split_manifest.json",
    SPLIT_DIR / "train.csv",
    SPLIT_DIR / "validation.csv",
    SPLIT_DIR / "holdout_test.csv",
    OUTPUT_DIR / "synthetic_pilot_screening.csv",
    OUTPUT_DIR / "synthetic_statistics.json",
]
missing = [str(p) for p in expected_artifacts if not p.exists()]
if missing:
    raise AssertionError(f"缺少产物：{missing}")
print(json.dumps({
    "artifact_count": len(expected_artifacts),
    "missing": missing,
    "cleaned_rows": int(len(cleaned_df)),
    "baseline_status": baseline_status["status"],
    "synthetic_smoke": synthetic_smoke_summary,
    "next_steps": [
        "在 RTX 5090 的官方 INF-VLLM Profile 中完成 smoke Baseline，并记录模型 revision、vLLM 版本和显存。",
        "补充 Equation Numeric Prior Specification 后，再实现 Generator、Identifiability 和 Structural OOD。",
        "加载 Qwen3 Tokenizer 后补充真实 token 长度统计。",
    ],
}, ensure_ascii=False, indent=2))
